# dcleaner demo

`dcleaner` is a data-cleaning toolkit on top of pandas. You say what you want;
it works out the rest — and every transform returns a **new** object, so
nothing you already computed changes under you.

Install: `pip install dcleaner`

A deliberately messy dataset ships inside the package, so this notebook runs
as-is with no downloads.


In [ ]:
from dclean import Data
import dclean

Data.samples()


## 1. What am I even looking at?

`report()` profiles the file before you touch it: dtypes, nulls, unique counts,
duplicates, summary stats — and the problems worth knowing about.


In [ ]:
dclean.report("sample_sales.csv")


The warnings tell the story: padded column names, prices stored as text
(`$1,234.50`), an entirely empty column, and duplicate rows.

## 2. Clean it — one call

`clean()` fixes all of that and prints exactly what it did, so nothing happens
behind your back.


In [ ]:
d = Data("sample_sales.csv").clean()
d.head()


Column names normalized, `$` and commas stripped, dates parsed, the empty
column gone, duplicates removed:


In [ ]:
d.dtypes()


## 3. Transforms never mutate

The object you call a transform on is left exactly as it was — the pandas
contract. Intermediate variables keep the value you gave them.


In [ ]:
raw = Data("sample_sales.csv")
cleaned = raw.clean(verbose=False)

print("raw     :", len(raw), "rows,", raw.to_df().shape[1], "cols")
print("cleaned :", len(cleaned), "rows,", cleaned.to_df().shape[1], "cols")
print("raw is unchanged:", raw.to_df().shape == (64, 9))


## 4. Ask a question — one call each

No separate `groupby` step, no dummy column, no escaping to raw pandas.


In [ ]:
d.mean("unit_price", by="city").print()


In [ ]:
d.count(by="region").print()      # rows per group
d.top(3, "unit_price").keep("order_id", "city", "unit_price").print()


`region` has both `West` and `west`. `clean()` deliberately does **not**
merge category spellings — that can be semantically wrong — but it is one
`mutate` away:


In [ ]:
d = d.mutate(region="region.str.lower()")
d.counts("region").print()


## 5. Derive, aggregate, plot

`plot()` infers `x` and `y` from a grouped aggregate, so the chain just ends.


In [ ]:
(d.mutate(revenue="units * unit_price")
   .sum("revenue", by="city")
   .plot("bar", title="Revenue by city")
   .savefig("revenue_by_city.png")
   .print())


## 6. What did all that actually do?

Because the toolkit makes decisions on your behalf, it records them.


In [ ]:
d.log()


## 7. Drop back to pandas whenever you want


In [ ]:
raw_df = d.to_df()
print(type(raw_df))
raw_df.describe()
